# 7. Regression

Predict the exact NewsGuard score 0–100 (ridge; ModernBERT with squared error on score/100). Same split, same models for both tasks. Linear models: the penalty is chosen with 3 inner folds of the
training outlets. ModernBERT: 5 epochs, lr 3e-5, effective batch 32, last epoch kept. For every test article the
outputs are saved in `predictions/regression/<model>/`, and every fold model in `Modelli/20_FromScratch/`.

In [1]:
TASK = "regression"
MODELS = "TF-IDF,Qwen3-8B,Mistral-24B"     # "ModernBERT" runs on the GPU
N_JOBS = 16

In [2]:
# Parameters
MODELS = "TF-IDF"


In [3]:
import warnings

import numpy as np
import pandas as pd

import common as C

warnings.filterwarnings("ignore")
pd.set_option("display.width", 200); pd.set_option("display.max_colwidth", 90)
C.OUT.mkdir(parents=True, exist_ok=True)
LOG = C.OUT / "log.txt"
print(C.environment())

import models as MO

art, outlets = MO.load()

{'python': '3.12.3', 'seed': 42, 'split seed': 1, 'numpy': '2.5.4', 'pandas': '3.0.6', 'sklearn': '1.9.1', 'scipy': '1.18.1', 'torch': '2.5.1+cu121', 'transformers': '4.51.3', 'bitsandbytes': '0.50.2', 'trafilatura': '2.3.1'}


/home/fabio.pinelli/John/johnVenv/lib/python3.12/site-packages/torch/cuda/__init__.py:716: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")


In [4]:
for name in MODELS.split(","):
    for k in range(1, 6):
        if name == "ModernBERT":
            MO.bert_fold(TASK, art, k, "answerdotai/ModernBERT-base", name, LOG)
        else:
            MO.linear_fold(TASK, name, art, k, X=None if name == "TF-IDF" else MO.load_embeddings(name, art),
                           n_jobs=N_JOBS, logfile=LOG)

2026-10-30 18:20:48  regression, TF-IDF, fold 1: penalty 0.3 (inside the grid), 120 s


2026-10-30 18:22:46  regression, TF-IDF, fold 2: penalty 1.0 (on the grid edge), 118 s


2026-10-30 18:24:44  regression, TF-IDF, fold 3: penalty 0.003 (inside the grid), 118 s


2026-10-30 18:26:42  regression, TF-IDF, fold 4: penalty 0.03 (inside the grid), 118 s


2026-10-30 18:28:36  regression, TF-IDF, fold 5: penalty 0.3 (inside the grid), 114 s


In [5]:
for name in MODELS.split(","):
    folder = C.PREDICTIONS / TASK / name
    if name == "ModernBERT":
        print(pd.concat([pd.read_csv(f).assign(fold=f.stem[-1]) for f in sorted(folder.glob("loss_fold*.csv"))])
              .pivot(index="epoch", columns="fold", values="loss").round(4))
    else:
        p = pd.concat([pd.read_csv(f) for f in sorted(folder.glob("penalty_fold*.csv"))])
        print(name, "chosen penalty per fold:", p[p.chosen].set_index("fold").penalty.to_dict())

TF-IDF chosen penalty per fold: {1: 0.3, 2: 1.0, 3: 0.003, 4: 0.03, 5: 0.3}
